# NFNet — Normalizer-Free Networks

**Paper**: Brock, A., De, S., Smith, S. L., & Simonyan, K. (2021).  
**"High-Performance Large-Scale Image Recognition Without Normalization"**  
ICML 2021 · arXiv:2102.06171  
DeepMind

---

## Why This Paper Matters

BatchNorm is everywhere in deep learning — it stabilises training and acts as implicit regularisation. But it has well-known pathologies:
- **Train/eval discrepancy**: BN uses batch statistics at train time but running averages at eval.
- **Small-batch instability**: batch statistics become noisy with batch size < 16.
- **Incompatibility with some architectures**: recurrent nets, certain distributed training setups.

NFNet proved none of this is necessary. NFNet-F5 reached **89.2% top-1 on ImageNet** — matching the best EfficientNets — with **zero BatchNorm layers**.  

The key: replace BN with **Scaled Weight Standardisation** and fix the residual variance explosion with a careful scaling scheme.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

plt.style.use('dark_background')
ACCENT = '#7EB8F7'
RED    = '#F77E7E'
GREEN  = '#7EF7A4'
GOLD   = '#F7C97E'

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('NFNet Architecture: Three Core Innovations', fontsize=14, color='white', y=1.02)

# --- Panel 1: Variance growth with/without NFNet fix ---
ax = axes[0]
blocks = np.arange(1, 25)
# Standard residual: Var(y_k) = k * Var(residual_branch) + Var(x_0)
var_std  = 1.0 + 0.04 * blocks  # grows linearly
var_nfnet = np.ones_like(blocks, dtype=float)  # stays near 1
ax.plot(blocks, var_std,   color=RED,   lw=2, label='Standard residual (variance grows)')
ax.plot(blocks, var_nfnet, color=GREEN, lw=2, label='NFNet (variance preserved)')
ax.set_xlabel('Block depth', color='white')
ax.set_ylabel('Activation variance', color='white')
ax.set_title('Variance Preservation', color=ACCENT)
ax.legend(fontsize=8)
ax.tick_params(colors='white')
for spine in ax.spines.values(): spine.set_color('#444')

# --- Panel 2: NFBlock structure (visual flow) ---
ax = axes[1]
ax.set_xlim(0, 4)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('NFBlock Structure', color=ACCENT)

steps = [
    (2, 9.2, 'Input x', '#555'),
    (2, 8.1, 'x / beta  (normalise var)', GOLD),
    (2, 6.9, 'WSConv 1×1  (GELU)', ACCENT),
    (2, 5.7, 'WSConv 3×3 groups  (GELU)', ACCENT),
    (2, 4.5, 'WSConv 1×1', ACCENT),
    (2, 3.3, 'Squeeze-Excite', GREEN),
    (2, 2.1, '× alpha  (scale residual)', GOLD),
    (2, 1.0, '+ shortcut  → output', '#555'),
]
for x, y, label, col in steps:
    ax.text(x, y, label, ha='center', va='center', fontsize=9,
            color='black', bbox=dict(boxstyle='round,pad=0.3', facecolor=col, alpha=0.85))

# Draw arrows
ys = [s[1] for s in steps]
for y0, y1 in zip(ys[:-1], ys[1:]):
    ax.annotate('', xy=(2, y1 + 0.3), xytext=(2, y0 - 0.3),
                arrowprops=dict(arrowstyle='->', color='#888'))

# --- Panel 3: Adaptive Gradient Clipping vs standard ---
ax = axes[2]
grad_norms = np.linspace(0, 5, 200)
param_norm = 1.0
lambda_    = 0.01
# AGC: clip to lambda * ||w|| / ||g|| * ||g|| = lambda * ||w||
agc_clip   = np.minimum(grad_norms, param_norm * lambda_)
# Standard global clip at threshold=0.01
std_clip   = np.minimum(grad_norms, 0.01)
ax.plot(grad_norms, agc_clip, color=GREEN, lw=2, label='AGC (per-parameter adaptive)')
ax.plot(grad_norms, std_clip, color=RED,   lw=2, linestyle='--', label='Standard clip (global threshold)')
ax.set_xlabel('||grad||', color='white')
ax.set_ylabel('Effective gradient norm', color='white')
ax.set_title('Adaptive Gradient Clipping', color=ACCENT)
ax.legend(fontsize=8)
ax.tick_params(colors='white')
for spine in ax.spines.values(): spine.set_color('#444')

plt.tight_layout()
plt.show()

## Mathematical Formulation

### 1. Scaled Weight Standardisation (SWS)

For a convolutional weight tensor $W \in \mathbb{R}^{C_{out} \times C_{in} \times k \times k}$:

$$\hat{W}_i = \frac{W_i - \mu_i}{\sigma_i + \epsilon} \cdot \frac{\gamma_i}{\sqrt{\text{fan\_in}}}$$

where $\mu_i = \mathbb{E}[W_i]$, $\sigma_i = \text{Std}(W_i)$ over the $C_{in} \times k \times k$ dims, and $\gamma_i$ is a learnable per-output-channel gain parameter.  
This ensures that each filter's convolution output has approximately **unit variance** under a normalised input.

### 2. Variance Preservation in Residual Blocks

Standard residuals accumulate variance: $\text{Var}(y_k) = k\,\text{Var}(F) + \text{Var}(x_0)$ (grows with depth).

NFNet corrects this with:
$$y_k = x_k + \alpha \cdot F\!\left(\frac{x_k}{\beta_k}\right)$$

where $\beta_k = \sqrt{\text{Var}(x_k)} \approx \sqrt{1 + \alpha^2(k-1)}$ (accumulated std), and $\alpha \approx 0.2$ (learnable scalar).  
The normalised input $x_k/\beta_k$ has unit variance; the $\alpha$ scale ensures $\text{Var}(y_k) \approx \text{Var}(x_k)$.

### 3. Adaptive Gradient Clipping (AGC)

For each parameter $\theta$:
$$g^* = g \cdot \min\!\left(1,\;\frac{\lambda \,\|\theta\|_F}{\|g\|_F}\right)$$

where $\|\cdot\|_F$ is the Frobenius norm computed **per-layer** (not globally), and $\lambda = 0.01$ is the clipping coefficient.  
This prevents any layer's gradient from exceeding a fraction $\lambda$ of its weight norm.

In [ ]:
"""WSConv2d and NFBlock code walkthrough."""
import sys
sys.path.insert(0, '../../../src')

import torch
import torch.nn as nn
import math

# ── Key mechanism: Scaled Weight Standardisation ──────────────────────────────
class WSConv2dDemo(nn.Conv2d):
    """Minimal demo of Scaled Weight Standardisation."""
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.gain = nn.Parameter(torch.ones(self.out_channels, 1, 1, 1))

    def forward(self, x):
        w = self.weight
        # Subtract per-filter mean
        w = w - w.mean(dim=[1, 2, 3], keepdim=True)
        # Normalise variance
        w = w / (w.var(dim=[1, 2, 3], keepdim=True, unbiased=False) + 1e-5).sqrt()
        # Scale to maintain unit-variance output
        fan_in = w[0].numel()
        w = w * (self.gain / math.sqrt(fan_in))
        import torch.nn.functional as F
        return F.conv2d(x, w, self.bias, self.stride, self.padding, self.dilation, self.groups)

# Verify: output should have variance ≈ 1 for unit-variance input
conv = WSConv2dDemo(3, 64, 3, padding=1)
x = torch.randn(8, 3, 32, 32)  # unit-variance input
y = conv(x)
print(f'Input  variance: {x.var():.4f}  (target 1.0)')
print(f'Output variance: {y.var():.4f}  (should be ≈ 1.0)')

# ── Demonstrate variance preservation ────────────────────────────────────────
print('\nVariance accumulation comparison:')
print('Block  Standard  NFNet-corrected')
var_std   = 1.0
var_nfnet = 1.0
alpha = 0.2
for k in range(1, 13):
    var_std   += 0.04            # alpha^2 = 0.04 per block
    var_nfnet += 0.0             # NFNet keeps variance constant
    print(f'  {k:2d}     {var_std:.3f}     {var_nfnet:.3f}')

In [ ]:
"""NFNet backbone usage example."""
import torch
from hale_core.nn.vision import build_vision_backbone, list_vision_backbones

# List all NFNet variants
nf_backbones = [n for n in list_vision_backbones() if 'nfnet' in n]
print('Registered NFNet variants:', nf_backbones)

# Build NFNet-F0 (smallest)
model = build_vision_backbone('nfnet_f0', img_size=224)
model.eval()

x = torch.randn(2, 3, 224, 224)
with torch.no_grad():
    out = model(x)

print(f'\nNFNet-F0 outputs:')
print(f'  pooled_output: {out.pooled_output.shape}  (should be [2, 1536])')
print(f'  feature_maps:  {len(out.feature_maps)} scales')
for i, fm in enumerate(out.feature_maps):
    print(f'    stage {i}: {tuple(fm.shape)}')

print(f'\nTotal parameters: {model.num_parameters():,}')
print(f'Trainable params: {model.num_parameters(trainable_only=True):,}')

# Demonstrate AGC (called after loss.backward() in training)
from hale_core.nn.vision.cnn.nfnet import agc_clip
y = out.pooled_output.sum()
y.backward()
print('\nBefore AGC: max grad norm =', max(
    p.grad.norm().item() for p in model.parameters() if p.grad is not None
))
agc_clip(model, lambda_=0.01)
print('After  AGC: max grad norm =', max(
    p.grad.norm().item() for p in model.parameters() if p.grad is not None
))

In [ ]:
"""Compare NFNet variants: parameters vs depth."""
import matplotlib.pyplot as plt
import torch
from hale_core.nn.vision import build_vision_backbone

plt.style.use('dark_background')
ACCENT = '#7EB8F7'

variants = ['nfnet_f0', 'nfnet_f1', 'nfnet_f2', 'nfnet_f3']
labels   = ['F0', 'F1', 'F2', 'F3']
depths   = [12, 24, 36, 48]  # sum of block counts
params   = []

for name in variants:
    m = build_vision_backbone(name, img_size=224)
    params.append(m.num_parameters() / 1e6)

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(labels, params, color=ACCENT, alpha=0.85, edgecolor='#222')
for bar, p in zip(bars, params):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{p:.0f}M', ha='center', color='white', fontsize=10)
ax.set_ylabel('Parameters (M)', color='white')
ax.set_title('NFNet Variant Sizes', color=ACCENT)
ax.tick_params(colors='white')
for spine in ax.spines.values(): spine.set_color('#444')
ax.set_facecolor('#1a1a2e')
fig.patch.set_facecolor('#0d0d1a')
plt.tight_layout()
plt.show()

## References

1. **Brock, A., De, S., Smith, S. L., & Simonyan, K. (2021).**  
   High-Performance Large-Scale Image Recognition Without Normalization.  
   *ICML 2021.* arXiv:2102.06171

2. **Brock, A., De, S., & Smith, S. L. (2021).**  
   Characterizing signal propagation to close the performance gap in unnormalized ResNets.  
   *ICLR 2021.* arXiv:2101.08692

3. **Ioffe, S., & Szegedy, C. (2015).**  
   Batch normalization: Accelerating deep network training by reducing internal covariate shift.  
   *ICML 2015.* arXiv:1502.03167

4. **Huang, G., Sun, Y., Liu, Z., Sedra, D., & Weinberger, K. Q. (2016).**  
   Deep networks with stochastic depth.  
   *ECCV 2016.* arXiv:1603.09382

5. **Hu, J., Shen, L., & Sun, G. (2018).**  
   Squeeze-and-Excitation Networks. *CVPR 2018.* arXiv:1709.01507

6. **Qiao, S., Wang, H., Liu, C., Shen, W., & Yuille, A. (2019).**  
   Weight standardization.  arXiv:1903.10520